In [1]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [2]:
df_path = "dataset"

In [3]:
data_set = df_path

In [4]:
import os
all_data = os.listdir(data_set)

In [5]:
all_data[:5]

['201311077', '201311192', '213311023', '221311076', '221311105']

In [6]:
len(all_data)

47

In [7]:
import os

# Get unique extensions from the dataset directory
extensions = set()
for root, dirs, files in os.walk(data_set):
    for file in files:
        extensions.add(os.path.splitext(file)[1].lower())

print(f"Unique file types found: {extensions}")

Unique file types found: {'.jpg', '.jpeg', '.dng', '.heic'}


In [8]:
from collections import Counter
import os

# count images types
exts = ['.dng', '.jpg', '.heic', '.jpeg']
counts = Counter(os.path.splitext(f)[1].lower() for r, d, files in os.walk(data_set) for f in files if os.path.splitext(f)[1].lower() in exts)

print(f"Counts: {dict(counts)}\nTotal: {sum(counts.values())}")

Counts: {'.jpg': 133, '.jpeg': 98, '.dng': 3, '.heic': 4}
Total: 238


In [9]:
import os
from PIL import Image
from pillow_heif import register_heif_opener

# Register HEIF opener for HEIC files
register_heif_opener()

output_dataset = "dataset_v2"

# All formats to convert (everything except .jpg)
formats_to_convert = ['.png', '.dng', '.heic', '.tiff', '.bmp', '.jpeg', '.gif', '.webp', '.tif']

print("Converting all non-JPG images to dataset_v2...")
converted_count = 0
skipped_count = 0

# Create output directory structure first
for folder in os.listdir(data_set):
    src_folder = os.path.join(data_set, folder)
    dst_folder = os.path.join(output_dataset, folder)
    if os.path.isdir(src_folder):
        os.makedirs(dst_folder, exist_ok=True)

# Convert all non-JPG images
for root, _, files in os.walk(data_set):
    for f in files:
        file_ext = os.path.splitext(f)[1].lower()
        
        # Skip if already JPG or not an image format we want to convert
        if file_ext == '.jpg':
            continue
            
        if file_ext in formats_to_convert:
            src_path = os.path.join(root, f)
            filename = os.path.splitext(f)[0]
            relative_path = os.path.relpath(root, data_set)
            dst_dir = os.path.join(output_dataset, relative_path)
            output_path = os.path.join(dst_dir, f"{filename}.jpg")
            
            # Skip if already converted
            if os.path.exists(output_path):
                print(f"Already exists: {filename}.jpg (skipped)")
                skipped_count += 1
                continue
            
            try:
                with Image.open(src_path) as img:
                    # Convert to RGB if needed
                    if img.mode != 'RGB':
                        img = img.convert('RGB')
                    
                    img.save(output_path, 'JPEG', quality=95)
                    converted_count += 1
                    print(f"✓ Converted: {f} → {filename}.jpg")
            except Exception as e:
                print(f"✗ Failed to convert {f}: {e}")

print(f"\nConversion Summary:")
print(f"  Converted: {converted_count} images")
print(f"  Skipped (already exists): {skipped_count} images")
print(f"  Total: {converted_count + skipped_count} images")

Converting all non-JPG images to dataset_v2...
✓ Converted: first_image.jpeg → first_image.jpg
✓ Converted: Second_image .jpeg → Second_image .jpg
✓ Converted: Third_image .jpeg → Third_image .jpg
✓ Converted: 1.jpeg → 1.jpg
✓ Converted: 2.jpeg → 2.jpg
✓ Converted: 3.jpeg → 3.jpg
✓ Converted: 4.jpeg → 4.jpg
✓ Converted: 5.jpeg → 5.jpg
✓ Converted: 6.jpeg → 6.jpg
✓ Converted: IMG_9056.DNG → IMG_9056.jpg
✓ Converted: IMG_9057.DNG → IMG_9057.jpg
✓ Converted: IMG_9058.DNG → IMG_9058.jpg
✓ Converted: img1.jpeg → img1.jpg
✓ Converted: img2.jpeg → img2.jpg
✓ Converted: img3.jpeg → img3.jpg
✓ Converted: img4.jpeg → img4.jpg
✓ Converted: img5.jpeg → img5.jpg
✓ Converted: image1.jpeg → image1.jpg
✓ Converted: image2.jpeg → image2.jpg
✓ Converted: image3.jpeg → image3.jpg
✓ Converted: image4.jpeg → image4.jpg
✓ Converted: image5.jpeg → image5.jpg
✓ Converted: 1.jpeg → 1.jpg
✓ Converted: 2.jpeg → 2.jpg
✓ Converted: 3.jpeg → 3.jpg
✓ Converted: 4.jpeg → 4.jpg
✓ Converted: 5.jpeg → 5.jpg
✓ Converted:

In [10]:
import os
import shutil
import numpy as np
from PIL import Image, ImageEnhance
import random

dataset_v2 = "dataset_v2"

# Create output directory structure
for folder in os.listdir(data_set):
    src_folder = os.path.join(data_set, folder)
    dst_folder = os.path.join(dataset_v2, folder)
    if os.path.isdir(src_folder) and not os.path.exists(dst_folder):
        os.makedirs(dst_folder)

def augment_and_save(img_path, output_dir):
    try:
        with Image.open(img_path) as img:
            img = img.convert('RGB')
            name = os.path.basename(os.path.splitext(img_path)[0])
            
            # Save original as JPG
            original_output = os.path.join(output_dir, f"{name}.jpg")
            img.save(original_output, 'JPEG')

            # 1. Random Rotation
            img.rotate(random.randint(-30, 30)).save(os.path.join(output_dir, f"{name}_rot.jpg"), 'JPEG')

            # 2. Random Horizontal Translation
            w, h = img.size
            shift = random.randint(-w//5, w//5)
            img.transform(img.size, Image.AFFINE, (1, 0, shift, 0, 1, 0)).save(os.path.join(output_dir, f"{name}_trans.jpg"), 'JPEG')

            # 3. Gaussian Noise Injection
            arr = np.array(img).astype(np.float64)
            noise = np.random.normal(0, 25, arr.shape)
            noisy_img = Image.fromarray(np.clip(arr + noise, 0, 255).astype(np.uint8))
            noisy_img.save(os.path.join(output_dir, f"{name}_noise.jpg"), 'JPEG')

            # 4. Random Sharpness Adjustment
            enhancer = ImageEnhance.Sharpness(img)
            enhancer.enhance(random.uniform(0.5, 2.0)).save(os.path.join(output_dir, f"{name}_sharp.jpg"), 'JPEG')
            
            print(f"Processed: {name}")
    except Exception as e:
        print(f"Skipping {img_path}: {e}")

# Apply to dataset
print("Starting augmentation and saving to dataset_v2...")
for root, _, files in os.walk(data_set):
    for f in files:
        if f.lower().endswith(('.jpg', '.jpeg')):
            src_path = os.path.join(root, f)
            relative_path = os.path.relpath(root, data_set)
            dst_dir = os.path.join(dataset_v2, relative_path)
            
            if not os.path.exists(dst_dir):
                os.makedirs(dst_dir)
            
            augment_and_save(src_path, dst_dir)

print("Augmentation complete. All files saved to dataset_v2.")

Starting augmentation and saving to dataset_v2...
Processed: 1st
Processed: 2nd
Processed: 3rd
Processed: 4th
Processed: 5th
Processed: fifth_image 
Processed: first_image
Processed: fourth_image
Processed: Second_image 
Processed: Third_image 
Processed: first_image
Processed: second_image
Processed: third_image
Processed: 1
Processed: 2
Processed: 3
Processed: 4
Processed: 5
Processed: 6
Processed: 1st
Processed: 2nd
Processed: 3rd
Processed: 4th
Processed: 5th
Processed: IMG_1
Processed: IMG_2
Processed: IMG_3
Processed: IMG_4
Processed: IMG 2
Processed: IMG 3
Processed: IMG 4
Processed: IMG1
Processed: 1
Processed: 2
Processed: 3
Processed: 4
Processed: 5
Processed: img1
Processed: img2
Processed: img3
Processed: img4
Processed: img5
Processed: img1
Processed: img2
Processed: img3
Processed: img4
Processed: img5
Processed: image1
Processed: image2
Processed: image3
Processed: image4
Processed: image5
Processed: IMG_20260505_230418_1
Processed: IMG_20260505_230429_1
Processed: IMG_2